In [ ]:
# 必要なライブラリのインストール
!pip install matplotlib-venn torchinfo transformers

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from matplotlib_venn import venn2
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import os
from transformers import ViTForImageClassification

# --- 1. Google Driveの設定 ---
print("Mounting Google Drive...")

# 保存先フォルダ（MyDrive直下の 'Research_Experiment' フォルダに保存します）
# 必要に応じて変更してください
SAVE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

# 保存ファイルパス
RESULT_JSON_PATH = os.path.join(SAVE_DIR, "cifar100_difficulty_labels.json")
GRAPH_IMAGE_PATH = os.path.join(SAVE_DIR, "difficulty_venn_diagram.png")

print(f"Results will be saved to: {SAVE_DIR}")

# --- 2. 設定 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

FORCE_INFERENCE = False  # Trueなら既存ファイルがあっても強制的に再推論

# --- 3. データセット準備 ---
print("Preparing Dataset...")

# Lowモデル用 (32x32, CIFAR標準正規化)
transform_low = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
])

# Highモデル用 (224x224, ImageNet標準正規化)
transform_high = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 生データ (画像ID管理用)
testset_raw = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=None)

# 修正: PIL画像をリストのまま受け取るための設定
def custom_collate(batch):
    images = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])
    return images, labels

testloader = DataLoader(testset_raw, batch_size=50, shuffle=False, collate_fn=custom_collate)

# --- 4. 推論実行関数 ---
def run_inference():
    print("Loading Models...")

    # Low: MobileNetV2 x0.5
    try:
        low_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_mobilenetv2_x0_5", pretrained=True)
    except:
        print("GitHub load failed. Using local fallback.")
        return []
    low_model.to(device).eval()

    # High: ViT-Base
    try:
        high_model = ViTForImageClassification.from_pretrained("Ahmed9275/Vit-Cifar100")
    except:
        print("HuggingFace load failed.")
        return []
    high_model.to(device).eval()

    results = []
    print("Starting Inference on CIFAR-100 Testset (10,000 images)...")

    # バッチ処理で推論
    global_idx = 0
    with torch.no_grad():
        for batch in tqdm(testloader):
            images, labels = batch # ここでのimagesはPIL形式のリスト
            labels = labels.to(device)

            # --- Lowモデル推論 ---
            # PIL -> Tensor変換
            batch_low = torch.stack([transform_low(img) for img in images]).to(device)
            out_low = low_model(batch_low)
            _, pred_low = torch.max(out_low, 1)

            # --- Highモデル推論 ---
            batch_high = torch.stack([transform_high(img) for img in images]).to(device)
            out_high = high_model(batch_high).logits
            _, pred_high = torch.max(out_high, 1)

            # 結果格納
            for i in range(len(labels)):
                label_val = labels[i].item()
                low_val = pred_low[i].item()
                high_val = pred_high[i].item()

                is_low_correct = (low_val == label_val)
                is_high_correct = (high_val == label_val)

                # 難易度カテゴリの決定
                if is_low_correct and is_high_correct:
                    cat = "Easy"
                elif not is_low_correct and is_high_correct:
                    cat = "Hard"
                elif not is_low_correct and not is_high_correct:
                    cat = "Impossible"
                else:
                    cat = "Inverse" # Lowのみ正解

                results.append({
                    "index": global_idx,
                    "label": label_val,
                    "low_pred": low_val,
                    "high_pred": high_val,
                    "low_correct": is_low_correct,
                    "high_correct": is_high_correct,
                    "category": cat
                })
                global_idx += 1

    return results

# --- 5. メイン処理 ---

if os.path.exists(RESULT_JSON_PATH) and not FORCE_INFERENCE:
    print(f"Loading existing results from '{RESULT_JSON_PATH}'...")
    with open(RESULT_JSON_PATH, 'r') as f:
        inference_data = json.load(f)
else:
    inference_data = run_inference()
    if inference_data:
        print(f"Saving results to '{RESULT_JSON_PATH}'...")
        with open(RESULT_JSON_PATH, 'w') as f:
            json.dump(inference_data, f, indent=2)


In [ ]:
# 必要なライブラリのインストール
!pip install matplotlib-venn

import json
import os
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# --- 1. Google Driveの設定 ---
# 前のステップと同じパスを使用します

BASE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(BASE_DIR, exist_ok=True)
JSON_PATH = os.path.join(BASE_DIR, "cifar100_difficulty_labels.json")
OUTPUT_IMAGE_PATH = os.path.join(BASE_DIR, "step2_difficulty_venn.png")

# --- 2. JSONデータの読み込み ---
print(f"Loading data from: {JSON_PATH}")

if not os.path.exists(JSON_PATH):
    raise FileNotFoundError(f"File not found: {JSON_PATH}. Please run Step 1 first.")

with open(JSON_PATH, 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples.")

# --- 3. データの集計 ---
# カテゴリごとのカウント
counts = {
    "Easy": 0,       # A ∩ B (Low○ High○)
    "Hard": 0,       # ¬A ∩ B (Low× High○)
    "Impossible": 0, # ¬A ∩ ¬B (Low× High×) -> 今回は図示しない
    "Inverse": 0     # A ∩ ¬B (Low○ High×) -> 少ないはず
}

for item in data:
    cat = item.get("category", "")
    if cat in counts:
        counts[cat] += 1

total = len(data)
# ベン図に表示するサンプルの総数（Impossibleを除く）
venn_total = counts['Easy'] + counts['Hard'] + counts['Inverse']

print("\n=== Classification Result ===")
print(f"Total: {total}")
for k, v in counts.items():
    print(f"{k}: {v} ({v/total*100:.1f}%)")

# --- 4. ベン図の作成 ---
plt.figure(figsize=(10, 8))

# venn2の引数: subsets = (Ab, aB, AB)
# Ab: Group A only (Low only) -> Inverse -> A ∩ ¬B
# aB: Group B only (High only) -> Hard -> ¬A ∩ B
# AB: Both (Intersection) -> Easy -> A ∩ B
subset_sizes = (counts['Inverse'], counts['Hard'], counts['Easy'])
v = venn2(subsets=subset_sizes, set_labels=('A: Low Correct', 'B: High Correct'))

# 色とラベルのカスタマイズ
try:
    # --------------------------------------------------------------------------
    # ★ラベルの表示位置調整 & 論理式による記述
    # --------------------------------------------------------------------------

    # 1. Inverse (A ∩ ¬B) -> 赤 (例外・ノイズ)
    if v.get_patch_by_id('10'):
        v.get_patch_by_id('10').set_color('red')
        v.get_patch_by_id('10').set_alpha(0.6)

        lbl = v.get_label_by_id('10')
        # 論理式で表記
        lbl.set_text(f"$A \\cap \\neg B$\n{counts['Inverse']}\n({counts['Inverse']/total*100:.1f}%)")
        lbl.set_fontsize(12)

    # 2. Hard (¬A ∩ B) -> 青 (Highの価値)
    if v.get_patch_by_id('01'):
        v.get_patch_by_id('01').set_color('blue')
        v.get_patch_by_id('01').set_alpha(0.6)

        lbl = v.get_label_by_id('01')
        # 論理式で表記
        lbl.set_text(f"$\\neg A \\cap B$\n{counts['Hard']}\n({counts['Hard']/total*100:.1f}%)")
        lbl.set_fontsize(12)

    # 3. Easy (A ∩ B) -> 緑 (包含部分)
    if v.get_patch_by_id('11'):
        v.get_patch_by_id('11').set_color('green')
        v.get_patch_by_id('11').set_alpha(0.6)

        lbl = v.get_label_by_id('11')
        # 論理式で表記
        lbl.set_text(f"$A \\cap B$\n{counts['Easy']}\n({counts['Easy']/total*100:.1f}%)")
        lbl.set_fontsize(12)

except Exception as e:
    print(f"Error customizing venn diagram: {e}")

# タイトル
plt.title(r"Verification of Inclusion Hypothesis: $A \subset B$", fontsize=16)

# 補足情報の表示（Impossibleは枠外に小さく記載）
plt.text(0.8, -0.7,
         f"Excluded: Impossible ($\\neg A \\cap \\neg B$)\n{counts['Impossible']} samples ({counts['Impossible']/total*100:.1f}%)",
         fontsize=10, ha='right', color='gray')

# 保存と表示
plt.savefig(OUTPUT_IMAGE_PATH)
print(f"\nVenn diagram saved to: {OUTPUT_IMAGE_PATH}")
plt.show()


In [ ]:
# 必要なライブラリのインストール
!pip install scipy

import json
import os
import numpy as np
from scipy.stats import chi2_contingency, norm

# --- 1. Google Driveの設定 ---

BASE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(BASE_DIR, exist_ok=True)
JSON_PATH = os.path.join(BASE_DIR, "cifar100_difficulty_labels.json")

# --- 2. JSONデータの読み込み ---
print(f"Loading data from: {JSON_PATH}")

if not os.path.exists(JSON_PATH):
    raise FileNotFoundError(f"File not found: {JSON_PATH}. Please run Step 1 first.")

with open(JSON_PATH, 'r') as f:
    data = json.load(f)

total = len(data)
print(f"Loaded {total} samples.\n")

# --- 3. データの集計 ---
counts = {
    "Easy": 0,       # Low○ High○
    "Hard": 0,       # Low× High○
    "Impossible": 0, # Low× High×
    "Inverse": 0     # Low○ High×
}

for item in data:
    cat = item.get("category", "")
    if cat in counts:
        counts[cat] += 1

# 各カテゴリの割合
r_easy = counts['Easy'] / total
r_hard = counts['Hard'] / total
r_imp = counts['Impossible'] / total
r_inv = counts['Inverse'] / total

# Lowモデル全体の正解率 P(Low)
n_low_correct = counts['Easy'] + counts['Inverse']
p_low = n_low_correct / total

# Highモデル全体の正解率 P(High)
n_high_correct = counts['Easy'] + counts['Hard']
p_high = n_high_correct / total


# --- 4. 統計的検定 1: 独立性のカイ二乗検定 ---
# 観測されたクロス集計表 (Observed)
#           High〇          High×
# Low〇     Easy (Both)     Inverse (LowOnly)
# Low×     Hard (HighOnly) Impossible (Neither)
observed = np.array([
    [counts['Easy'], counts['Inverse']],
    [counts['Hard'], counts['Impossible']]
])

# カイ二乗検定
chi2, p_chi2, dof, expected = chi2_contingency(observed)


# --- 5. 統計的検定 2: Z検定 (正規近似) ---
# 帰無仮説: 「Inverseの発生率は、LowとHighが独立（ランダム）だった場合の期待値に従う」
# 期待されるInverse率 P_exp = P(Low) * P(not High)
p_exp_inverse = p_low * (1.0 - p_high)
n_exp_inverse = p_exp_inverse * total

# 観測されたInverse率 P_obs
p_obs_inverse = r_inv

# Zスコアの計算
# 標準誤差 SE = sqrt( p(1-p) / n )
se = np.sqrt(p_exp_inverse * (1.0 - p_exp_inverse) / total)
z_score = (p_obs_inverse - p_exp_inverse) / se

# p値 (片側検定: 観測値が期待値より低い確率)
p_val_z = norm.cdf(z_score)


# --- 6. レポート出力 ---
print("="*60)
print("             STATISTICAL VALIDATION REPORT             ")
print("="*60)

print("[1] Basic Statistics")
print(f"  Total Images       : {total}")
print(f"  Low Model Accuracy : {p_low*100:.2f}%")
print(f"  High Model Accuracy: {p_high*100:.2f}%")
print("-" * 60)
print(f"  Easy (A ∩ B)       : {counts['Easy']:>5} ({r_easy*100:.2f}%)")
print(f"  Hard (¬A ∩ B)      : {counts['Hard']:>5} ({r_hard*100:.2f}%)")
print(f"  Inverse (A ∩ ¬B)   : {counts['Inverse']:>5} ({r_inv*100:.2f}%)  <-- Critical Metric")
print(f"  Impossible         : {counts['Impossible']:>5} ({r_imp*100:.2f}%)")
print("="*60)

print("\n[2] Independence Test (Chi-Squared)")
print("  Null Hypothesis (H0): Low and High model predictions are independent.")
print(f"  Chi2 Value : {chi2:.4f}")
print(f"  p-value    : {p_chi2:.4e}")
if p_chi2 < 0.01:
    print("  >> RESULT: Significant (Rejected H0). Models are correlated.")
else:
    print("  >> RESULT: Not Significant.")
print("="*60)

print("\n[3] Inclusion Hypothesis Test (Z-Test)")
print("  Hypothesis: 'Inverse' cases are significantly fewer than random chance.")
print("  (Proof of Inclusion Relationship: Easy ⊂ Hard)")
print("-" * 60)
print(f"  Expected Inverse (if Random) : {n_exp_inverse:.1f} images ({p_exp_inverse*100:.2f}%)")
print(f"  Observed Inverse (Actual)    : {counts['Inverse']} images ({p_obs_inverse*100:.2f}%)")
print(f"  Difference                   : {counts['Inverse'] - n_exp_inverse:.1f} images")
print("-" * 60)
print(f"  Z-Score : {z_score:.4f} σ")
print(f"  p-value : {p_val_z:.4e}")

if z_score < -3.0:
    print("  >> CONCLUSION: HIGHLY SIGNIFICANT.")
    print(f"     The occurrence of Inverse samples is {abs(z_score):.1f} sigma lower than chance.")
    print("     This statistically proves the existence of difficulty hierarchy.")
else:
    print("  >> CONCLUSION: Not significant enough to prove strong inclusion.")
print("="*60)


In [ ]:
# --- 【完全実推論版】カスケード方式の性能検証スクリプト ---

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import json
import os

# 1. 初期設定

BASE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(BASE_DIR, exist_ok=True)
INPUT_JSON_PATH = os.path.join(BASE_DIR, "cifar100_difficulty_labels.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FLOPS_LOW  = 0.301 # MobileNetV2
FLOPS_HIGH = 17.6  # ViT-Base

# 2. ユーザー指定のTransforms
# Lowモデル用 (32x32, CIFAR標準正規化)
transform_low = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
])

# Highモデル用 (224x224, ImageNet標準正規化)
# ※今回はHighモデルの推論自体は回さず、JSONの正解フラグ(high_correct)を流用しますが、
# 実装に組み込む際はこちらを使用します。
transform_high = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 3. データのロード
print("Loading JSON Data...")
with open(INPUT_JSON_PATH, 'r') as f:
    data_list = json.load(f)

# Highモデル単独の精度（目標設定用）
acc_high_only = sum(1 for d in data_list if d["high_correct"]) / len(data_list) * 100
TARGET_ACCURACY = acc_high_only - 1.0

# 4. 【本物の推論】CIFAR-100学習済みMobileNetV2のダウンロードと推論
print("\nDownloading Pre-trained CIFAR-100 MobileNetV2...")
# chenyaofo氏が公開しているCIFAR専用の学習済みモデルを直接読み込みます
low_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_mobilenetv2_x1_0", pretrained=True)
low_model = low_model.to(device)
low_model.eval()

testset_raw = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=None)

print("Running Actual Inference for Low Model...")
with torch.no_grad():
    for item in tqdm(data_list, desc="Inference"):
        img_raw, label = testset_raw[item['index']]

        # 指定されたtransform_lowを適用
        img_tensor = transform_low(img_raw).unsqueeze(0).to(device)

        # 推論と確信度（Softmaxの最大値）の計算
        outputs = low_model(img_tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

        # 実際の推論結果でJSONのデータを上書き（完全な実データ化）
        item['real_low_conf'] = conf.item()
        item['real_low_correct'] = (pred.item() == label)

# 5. カスケード方式のシミュレーション（最適閾値探索）
print("\n" + "="*55)
print("     REAL CASCADE INFERENCE RESULTS     ")
print("="*55)

data_list.sort(key=lambda x: x["real_low_conf"], reverse=True)
total_n = len(data_list)
thresholds = np.linspace(0, 1.0, 201)

best_cost = float('inf')
best_th = None
best_acc = None

for th in thresholds:
    to_low_only = [d for d in data_list if d["real_low_conf"] >= th]
    to_high_added = [d for d in data_list if d["real_low_conf"] < th]

    n_low_only = len(to_low_only)
    n_high_added = len(to_high_added)

    # 【カスケードのコスト】全員がLowを実行し、閾値未満の画像はHighも追加実行（二重推論）
    total_cost = (total_n * FLOPS_LOW) + (n_high_added * FLOPS_HIGH)
    avg_cost = total_cost / total_n

    correct_low = sum(1 for d in to_low_only if d["real_low_correct"])
    correct_high = sum(1 for d in to_high_added if d["high_correct"])
    accuracy = 100 * (correct_low + correct_high) / total_n

    if accuracy >= TARGET_ACCURACY and avg_cost < best_cost:
        best_cost = avg_cost
        best_th = th
        best_acc = accuracy

# 6. 結果出力
print(f"Target Accuracy: >= {TARGET_ACCURACY:.2f}% (High Only: {acc_high_only:.2f}%)")

if best_th is not None:
    n_easy_total = sum(1 for d in data_list if d["real_low_correct"])
    n_hard_total = sum(1 for d in data_list if not d["real_low_correct"])

    n_easy_to_low = sum(1 for d in data_list if d["real_low_correct"] and d["real_low_conf"] >= best_th)
    n_easy_to_high = sum(1 for d in data_list if d["real_low_correct"] and d["real_low_conf"] < best_th)

    n_hard_to_low = sum(1 for d in data_list if not d["real_low_correct"] and d["real_low_conf"] >= best_th)
    n_hard_to_high = sum(1 for d in data_list if not d["real_low_correct"] and d["real_low_conf"] < best_th)

    print(f"\n[Cascade Architecture Performance]")
    print(f"System Average Cost : {best_cost:.3f} GFLOPs (Threshold: {best_th:.2f})")
    print(f"System Accuracy     : {best_acc:.2f}%")

    print(f"\n  [True Hard Samples] (Total: {n_hard_total})")
    print(f"    -> High追加実行 (Accuracy Maintained) : {n_hard_to_high:>5} ({n_hard_to_high/n_hard_total*100:.1f}%)")
    print(f"    -> Lowのみで終了 (Accuracy Loss)      : {n_hard_to_low:>5} ({n_hard_to_low/n_hard_total*100:.1f}%)")

    print(f"\n  [True Easy Samples] (Total: {n_easy_total})")
    print(f"    -> Lowのみで終了 (Energy Saved!)      : {n_easy_to_low:>5} ({n_easy_to_low/n_easy_total*100:.1f}%)")
    print(f"    -> Highも追加実行 (Energy Wasted)     : {n_easy_to_high:>5} ({n_easy_to_high/n_easy_total*100:.1f}%)")

    print("\n[Latency Analysis (Worst-case)]")
    print(f"  Hard画像に対する推論遅延: {FLOPS_LOW + FLOPS_HIGH:.3f} GFLOPs相当 (二重推論ペナルティ)")
    print(f"  二重推論が発生した割合  : {n_high_added / total_n * 100:.1f}% の画像で遅延増大")
else:
    print("\n[Result] Target accuracy could not be achieved.")

print("="*55)


In [ ]:
# --- 【完全実推論版】パラレル方式の性能検証スクリプト ---

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import json
import os

# 1. 初期設定

BASE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(BASE_DIR, exist_ok=True)
INPUT_JSON_PATH = os.path.join(BASE_DIR, "cifar100_difficulty_labels.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FLOPS_LOW  = 0.301 # MobileNetV2 (CNN)
FLOPS_HIGH = 17.6  # ViT-Base

# 【パラレル方式特有のパラメータ】
# CNNが「Easy」と判定してViTを強制停止させるまでに、ViTが消費してしまう電力（無駄骨）の割合
ALPHA_PENALTY = 0.10  # 仮にViTの計算が10%進んでしまうと設定

# 2. ユーザー指定のTransforms
transform_low = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
])

transform_high = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 3. データのロード
print("Loading JSON Data...")
with open(INPUT_JSON_PATH, 'r') as f:
    data_list = json.load(f)

# Highモデル単独の精度（目標設定用）
acc_high_only = sum(1 for d in data_list if d["high_correct"]) / len(data_list) * 100
TARGET_ACCURACY = acc_high_only - 1.0

# 4. 【本物の推論】CIFAR-100学習済みMobileNetV2のダウンロードと推論
print("\nDownloading Pre-trained CIFAR-100 MobileNetV2...")
low_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_mobilenetv2_x1_0", pretrained=True)
low_model = low_model.to(device)
low_model.eval()

testset_raw = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=None)

print("Running Actual Inference for Low Model...")
with torch.no_grad():
    for item in tqdm(data_list, desc="Inference"):
        img_raw, label = testset_raw[item['index']]
        img_tensor = transform_low(img_raw).unsqueeze(0).to(device)

        outputs = low_model(img_tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

        item['real_low_conf'] = conf.item()
        item['real_low_correct'] = (pred.item() == label)

# 5. パラレル方式のシミュレーション（最適閾値探索）
print("\n" + "="*60)
print("     REAL PARALLEL INFERENCE RESULTS     ")
print("="*60)

data_list.sort(key=lambda x: x["real_low_conf"], reverse=True)
total_n = len(data_list)
thresholds = np.linspace(0, 1.0, 201)

best_cost = float('inf')
best_th = None
best_acc = None

for th in thresholds:
    to_low_only = [d for d in data_list if d["real_low_conf"] >= th]
    to_high_added = [d for d in data_list if d["real_low_conf"] < th]

    n_low_only = len(to_low_only)
    n_high_added = len(to_high_added)

    # 【パラレル方式のコスト計算】
    # 画像が入った瞬間、CNNとViTが同時にスタートする。
    # 1. 全員がCNN(Low)を完走する: (total_n * FLOPS_LOW)
    # 2. Easy判定の画像は、ViTを途中で止めるが無駄は発生する: (n_low_only * ALPHA_PENALTY * FLOPS_HIGH)
    # 3. Hard判定の画像は、ViTを最後まで完走させる: (n_high_added * FLOPS_HIGH)
    total_cost = (total_n * FLOPS_LOW) + \
                 (n_low_only * (ALPHA_PENALTY * FLOPS_HIGH)) + \
                 (n_high_added * FLOPS_HIGH)

    avg_cost = total_cost / total_n

    correct_low = sum(1 for d in to_low_only if d["real_low_correct"])
    correct_high = sum(1 for d in to_high_added if d["high_correct"])
    accuracy = 100 * (correct_low + correct_high) / total_n

    if accuracy >= TARGET_ACCURACY and avg_cost < best_cost:
        best_cost = avg_cost
        best_th = th
        best_acc = accuracy

# 6. 結果出力
print(f"Target Accuracy: >= {TARGET_ACCURACY:.2f}% (High Only: {acc_high_only:.2f}%)")
print(f"ViT Abort Penalty (Alpha) : {ALPHA_PENALTY*100:.0f}% wasted per Easy image\n")

if best_th is not None:
    n_easy_total = sum(1 for d in data_list if d["real_low_correct"])
    n_hard_total = sum(1 for d in data_list if not d["real_low_correct"])

    n_easy_to_low = sum(1 for d in data_list if d["real_low_correct"] and d["real_low_conf"] >= best_th)
    n_easy_to_high = sum(1 for d in data_list if d["real_low_correct"] and d["real_low_conf"] < best_th)

    n_hard_to_low = sum(1 for d in data_list if not d["real_low_correct"] and d["real_low_conf"] >= best_th)
    n_hard_to_high = sum(1 for d in data_list if not d["real_low_correct"] and d["real_low_conf"] < best_th)

    print(f"[Parallel Architecture Performance]")
    print(f"System Average Cost : {best_cost:.3f} GFLOPs (Threshold: {best_th:.2f})")
    print(f"System Accuracy     : {best_acc:.2f}%")

    print(f"\n  [True Hard Samples] (Total: {n_hard_total})")
    print(f"    -> ViTフル稼働 (Accuracy Maintained) : {n_hard_to_high:>5} ({n_hard_to_high/n_hard_total*100:.1f}%)")
    print(f"    -> ViT途中停止 (Accuracy Loss)       : {n_hard_to_low:>5} ({n_hard_to_low/n_hard_total*100:.1f}%)")

    print(f"\n  [True Easy Samples] (Total: {n_easy_total})")
    print(f"    -> ViT途中停止 (Energy partially Saved) : {n_easy_to_low:>5} ({n_easy_to_low/n_easy_total*100:.1f}%)")
    print(f"    -> ViTフル稼働 (Energy completely Wasted): {n_easy_to_high:>5} ({n_easy_to_high/n_easy_total*100:.1f}%)")

    print("\n[Latency Analysis (Worst-case)]")
    print(f"  Hard画像に対する推論遅延: {max(FLOPS_LOW, FLOPS_HIGH):.3f} GFLOPs相当")
    print(f"  (カスケード方式のような {FLOPS_LOW + FLOPS_HIGH:.3f} の加算ペナルティは発生せず、最悪実行時間を保証)")
else:
    print("\n[Result] Target accuracy could not be achieved.")

print("="*60)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. 固定パラメータの設定
# ==========================================
C_H = 17.6     # ViT (Highモデル) の演算量 [GFLOPs]
C_L = 0.301    # CNN (Lowモデル) の演算量 [GFLOPs]
alpha = 0.10   # パラレル方式におけるViTの無駄骨割合 (10%)
C_r = 0.000    # ルーターの演算量 (遅延ゼロのストリーム処理のためほぼ0)

# ==========================================
# 2. 変数パラメータ（救出率 R）の設定
# ==========================================
# 救出率 R を 0.0 (0%) から 1.0 (100%) まで100段階で変化させる
R = np.linspace(0, 1.0, 100)

# ==========================================
# 3. 各アーキテクチャの平均演算コスト計算モデル
# ==========================================
# ① ViT Only: 常にC_H
cost_vit = np.full_like(R, C_H)

# ② Cascade: C_L + (1 - R) * C_H
cost_cascade = C_L + (1 - R) * C_H

# ③ Parallel: C_L + R * alpha * C_H + (1 - R) * C_H
cost_parallel = C_L + R * alpha * C_H + (1 - R) * C_H

# ④ Proposed Router: R * C_L + (1 - R) * C_H + C_r
cost_router = R * C_L + (1 - R) * C_H + C_r

# ==========================================
# 4. 損益分岐点（Breakeven Point）の算出
# ==========================================
# ルーターがパラレル方式のコストを下回る（逆転する）Rを計算
# R*C_L + (1-R)*C_H = C_L + R*alpha*C_H + (1-R)*C_H
# R*C_L = C_L + R*alpha*C_H  =>  R(C_L - alpha*C_H) = C_L
break_even_R = C_L / (alpha * C_H - C_L + 1e-9) # ゼロ除算防止
if break_even_R < 0 or break_even_R > 1:
    break_even_R = None # 交点がない場合

# ==========================================
# 5. グラフの描画
# ==========================================
plt.figure(figsize=(10, 6))

# 各方式のプロット
plt.plot(R * 100, cost_vit, label='ViT Only', color='black', linestyle=':')
plt.plot(R * 100, cost_cascade, label='Cascade', color='blue', linestyle='--')
plt.plot(R * 100, cost_parallel, label='Parallel', color='orange', linestyle='-.')
plt.plot(R * 100, cost_router, label='Proposed Router', color='red', linewidth=2)

# 現在のルーター性能 (20.9%) と目標地点 (損益分岐点) のハイライト
current_R = 0.209
current_cost = current_R * C_L + (1 - current_R) * C_H + C_r
plt.scatter([current_R * 100], [current_cost], color='red', s=100, zorder=5)
plt.text(current_R * 100 + 2, current_cost + 0.5, f'Current (R={current_R*100:.1f}%)', color='red')

if break_even_R:
    plt.axvline(x=break_even_R * 100, color='gray', linestyle='--', alpha=0.7)
    plt.scatter([break_even_R * 100], [cost_parallel[int(break_even_R*100)]], color='green', s=100, zorder=5)
    plt.text(break_even_R * 100 + 2, cost_parallel[int(break_even_R*100)] + 0.5, f'Breakeven vs Parallel (R={break_even_R*100:.1f}%)', color='green')

# グラフの装飾
plt.title('Average Computational Cost vs. Rescue Rate ($R$)', fontsize=14)
plt.xlabel('Rescue Rate $R$ (%)', fontsize=12)
plt.ylabel('Average Cost (GFLOPs)', fontsize=12)
plt.xlim(0, 60) # 0%〜60%の範囲を拡大表示（必要に応じて100に変えてください）
plt.ylim(0, 20)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.tight_layout()

# 表示
plt.show()

# ==========================================
# 6. コンソール出力（サマリー）
# ==========================================
print("=== Simulation Summary ===")
print(f"Fixed Parameters: C_H={C_H} GFLOPs, C_L={C_L} GFLOPs, Alpha={alpha*100}%")
if break_even_R:
    print(f"\n[Goal] To beat Parallel Architecture, Router needs R > {break_even_R*100:.2f}%")
print(f"Current Router (R=20.9%): {current_cost:.3f} GFLOPs")


In [ ]:
# --- 【方針② 完全版】8つの軽量統計量＋決定木ベースのルーター探索スクリプト ---

import numpy as np
import json
import os
import cv2
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
import torchvision

# --- 1. 初期設定 ---

BASE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(BASE_DIR, exist_ok=True)
INPUT_JSON_PATH = os.path.join(BASE_DIR, "cifar100_difficulty_labels.json")

FLOPS_LOW  = 0.301 # MobileNetV2
FLOPS_HIGH = 17.6  # ViT-Base
FLOPS_ROUTER = 0.000000 # 決定木（If文）のコストはFPGA上で実質ゼロ（DSP消費なし）

# --- 2. 軽量特徴量（FPGAでストリーム計算可能な8つの指標） ---
def extract_lightweight_features(img_pil):
    img = np.array(img_pil) # (32, 32, 3)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray_f = gray.astype(np.float32)
    h, w = gray.shape

    # 1. コントラスト（輝度の分散）
    variance = np.var(gray)

    # 2 & 3. エッジ強度平均 & エッジ密度
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_magnitude = np.sqrt(sobelx**2 + sobely**2)
    edge_mean = np.mean(edge_magnitude)
    edge_density = np.sum(edge_magnitude > 50) / gray.size

    # 4. 色の複雑さ（RGB分散の平均）
    color_var = (np.var(img[:,:,0]) + np.var(img[:,:,1]) + np.var(img[:,:,2])) / 3.0

    # 5. 白飛び・黒つぶれの割合 (Extreme Pixels)
    extreme_pixels = np.sum((gray < 15) | (gray > 240)) / gray.size

    # 6. 疑似エントロピー (Total Variation)
    tv = np.mean(np.abs(gray_f[:, 1:] - gray_f[:, :-1]))

    # 7. 中央と周辺の輝度差 (Center-Surround Difference)
    cy_min, cy_max = h // 4, 3 * h // 4
    cx_min, cx_max = w // 4, 3 * w // 4
    center_area = gray[cy_min:cy_max, cx_min:cx_max]
    center_sum = np.sum(center_area)
    total_sum = np.sum(gray)
    surround_mean = (total_sum - center_sum) / (gray.size - center_area.size)
    center_mean = np.mean(center_area)
    center_surround_diff = abs(center_mean - surround_mean)

    # 8. 平坦領域の割合 (Flatness)
    flatness = np.sum(np.abs(gray_f[:, 1:] - gray_f[:, :-1]) < 3) / (gray.size - h)

    return [variance, edge_mean, edge_density, color_var,
            extreme_pixels, tv, center_surround_diff, flatness]

# --- 3. データセットの読み込みと特徴量抽出 ---
print("Loading Data and Extracting 8 Features...")
testset_raw = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=None)

with open(INPUT_JSON_PATH, 'r') as f:
    data_list = json.load(f)

acc_high_only = sum(1 for d in data_list if d["high_correct"]) / len(data_list) * 100
TARGET_ACCURACY = acc_high_only - 1.0

features = []
labels = []

for item in tqdm(data_list, desc="Extracting Features"):
    idx = item['index']
    img_pil, _ = testset_raw[idx]
    feats = extract_lightweight_features(img_pil)
    features.append(feats)
    labels.append(1 if item['low_correct'] else 0)

X = np.array(features)
y = np.array(labels)

# --- 4. ランダムフォレストの学習 ---
print("\nTraining Random Forest Router (Zero-Latency Config)...")
# FPGAに落とし込めるレベルの浅い決定木（max_depth=6）を使用
clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
clf.fit(X, y)
probs = clf.predict_proba(X)

for i, item in enumerate(data_list):
    item['rf_conf'] = probs[i][1]

# --- 5. シミュレーション (最適閾値の探索) ---
data_list.sort(key=lambda x: x["rf_conf"], reverse=True)
total_n = len(data_list)
thresholds = np.linspace(0, 1.0, 201)

best_cost = float('inf')
best_th = None
best_acc = None

for th in thresholds:
    to_low = [d for d in data_list if d["rf_conf"] >= th]
    to_high = [d for d in data_list if d["rf_conf"] < th]

    n_low = len(to_low)
    n_high = len(to_high)

    # 【提案手法のコスト】ルーター(実質0) + 分岐先(Low or High)の片方のみ
    total_cost = (total_n * FLOPS_ROUTER) + (n_low * FLOPS_LOW) + (n_high * FLOPS_HIGH)
    avg_cost = total_cost / total_n

    correct_low = sum(1 for d in to_low if d["low_correct"])
    correct_high = sum(1 for d in to_high if d["high_correct"])
    accuracy = 100 * (correct_low + correct_high) / total_n

    if accuracy >= TARGET_ACCURACY and avg_cost < best_cost:
        best_cost = avg_cost
        best_th = th
        best_acc = accuracy

# --- 6. 詳細結果の出力 ---
print("\n" + "="*55)
print("     PROPOSED ROUTER (8 Features + Random Forest)     ")
print("="*55)
print(f"Target Accuracy: >= {TARGET_ACCURACY:.2f}% (High Only: {acc_high_only:.2f}%)")

if best_th is not None:
    n_easy_total = sum(1 for d in data_list if d["low_correct"])
    n_hard_total = sum(1 for d in data_list if not d["low_correct"])

    n_easy_to_low = sum(1 for d in data_list if d["low_correct"] and d["rf_conf"] >= best_th)
    n_easy_to_high = sum(1 for d in data_list if d["low_correct"] and d["rf_conf"] < best_th)

    n_hard_to_low = sum(1 for d in data_list if not d["low_correct"] and d["rf_conf"] >= best_th)
    n_hard_to_high = sum(1 for d in data_list if not d["low_correct"] and d["rf_conf"] < best_th)

    print(f"\n[System Average Performance]")
    print(f"Router Execution Cost : ~ 0.000 GFLOPs (Zero DSP utilization)")
    print(f"System Average Cost   : {best_cost:.3f} GFLOPs (Threshold: {best_th:.2f})")
    print(f"System Accuracy       : {best_acc:.2f}%")

    print(f"\n[Routing Details]")
    print(f"  [True Hard Samples] (Total: {n_hard_total})")
    print(f"    -> Sent to High (Accuracy Maintained) : {n_hard_to_high:>5} ({n_hard_to_high/n_hard_total*100:.1f}%)")
    print(f"    -> Sent to Low  (Accuracy Loss)       : {n_hard_to_low:>5} ({n_hard_to_low/n_hard_total*100:.1f}%)")

    print(f"  [True Easy Samples] (Total: {n_easy_total})")
    print(f"    -> Sent to Low  (Energy Saved!)       : {n_easy_to_low:>5} ({n_easy_to_low/n_easy_total*100:.1f}%)")
    print(f"    -> Sent to High (Energy Wasted)       : {n_easy_to_high:>5} ({n_easy_to_high/n_easy_total*100:.1f}%)")

    print("\n[Latency Analysis (Worst-case)]")
    print(f"  Hard画像に対する推論遅延: {FLOPS_HIGH:.3f} GFLOPs (ルーター遅延隠蔽により追加ペナルティなし)")
    print(f"  二重推論が発生した割合  : 0.0% (完全独立ルーティング)")

    print("\n[Feature Importances]")
    feature_names = [
        "1. Variance (Contrast)", "2. Edge Mean", "3. Edge Density", "4. Color Variance",
        "5. Extreme Pixels", "6. Total Variation", "7. Center-Surround Diff", "8. Flatness"
    ]
    importances = clf.feature_importances_
    # 重要度順にソートして表示
    sorted_idx = np.argsort(importances)[::-1]
    for idx in sorted_idx:
        print(f"  - {feature_names[idx]:<25}: {importances[idx]*100:>4.1f}%")

else:
    print("\n[Result] Target accuracy could not be achieved.")

print("="*55)


In [ ]:
import lightgbm as lgb

# --- 4. LightGBM（勾配ブースティング）の学習 ---
print("\nTraining LightGBM Router (Zero-Latency Config)...")

# FPGA制約（最大If文ネスト数）を守るため max_depth=6 に固定。
# LightGBMは葉(leaf)単位で成長するため、深さ6での最大葉数(2^6-1=63)を指定して表現力を最大化します。
clf = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    num_leaves=63,
    learning_rate=0.05, # 学習率を少し下げて慎重に学習させる
    random_state=42,
    n_jobs=-1,
    importance_type='gain' # どの特徴量が情報利得（精度向上）に貢献したかで評価
)

# 学習の実行
clf.fit(X, y)
probs = clf.predict_proba(X)

for i, item in enumerate(data_list):
    item['rf_conf'] = probs[i][1] # 変数名はrf_confのまま流用

# --- 5. シミュレーション (最適閾値の探索) ---
data_list.sort(key=lambda x: x["rf_conf"], reverse=True)
total_n = len(data_list)
thresholds = np.linspace(0, 1.0, 201)

best_cost = float('inf')
best_th = None
best_acc = None

for th in thresholds:
    to_low = [d for d in data_list if d["rf_conf"] >= th]
    to_high = [d for d in data_list if d["rf_conf"] < th]

    n_low = len(to_low)
    n_high = len(to_high)

    total_cost = (total_n * FLOPS_ROUTER) + (n_low * FLOPS_LOW) + (n_high * FLOPS_HIGH)
    avg_cost = total_cost / total_n

    correct_low = sum(1 for d in to_low if d["low_correct"])
    correct_high = sum(1 for d in to_high if d["high_correct"])
    accuracy = 100 * (correct_low + correct_high) / total_n

    if accuracy >= TARGET_ACCURACY and avg_cost < best_cost:
        best_cost = avg_cost
        best_th = th
        best_acc = accuracy

# --- 6. 詳細結果の出力 ---
print("\n" + "="*55)
print("     PROPOSED ROUTER (8 Features + LightGBM)     ")
print("="*55)
print(f"Target Accuracy: >= {TARGET_ACCURACY:.2f}% (High Only: {acc_high_only:.2f}%)")

if best_th is not None:
    n_easy_total = sum(1 for d in data_list if d["low_correct"])
    n_hard_total = sum(1 for d in data_list if not d["low_correct"])

    n_easy_to_low = sum(1 for d in data_list if d["low_correct"] and d["rf_conf"] >= best_th)
    n_easy_to_high = sum(1 for d in data_list if d["low_correct"] and d["rf_conf"] < best_th)

    n_hard_to_low = sum(1 for d in data_list if not d["low_correct"] and d["rf_conf"] >= best_th)
    n_hard_to_high = sum(1 for d in data_list if not d["low_correct"] and d["rf_conf"] < best_th)

    print(f"\n[System Average Performance]")
    print(f"Router Execution Cost : ~ 0.000 GFLOPs (Zero DSP utilization)")
    print(f"System Average Cost   : {best_cost:.3f} GFLOPs (Threshold: {best_th:.2f})")
    print(f"System Accuracy       : {best_acc:.2f}%")

    print(f"\n[Routing Details]")
    print(f"  [True Hard Samples] (Total: {n_hard_total})")
    print(f"    -> Sent to High (Accuracy Maintained) : {n_hard_to_high:>5} ({n_hard_to_high/n_hard_total*100:.1f}%)")
    print(f"    -> Sent to Low  (Accuracy Loss)       : {n_hard_to_low:>5} ({n_hard_to_low/n_hard_total*100:.1f}%)")

    print(f"  [True Easy Samples] (Total: {n_easy_total})")
    print(f"    -> Sent to Low  (Energy Saved!)       : {n_easy_to_low:>5} ({n_easy_to_low/n_easy_total*100:.1f}%)")
    print(f"    -> Sent to High (Energy Wasted)       : {n_easy_to_high:>5} ({n_easy_to_high/n_easy_total*100:.1f}%)")

    print("\n[Latency Analysis (Worst-case)]")
    print(f"  Hard画像に対する推論遅延: {FLOPS_HIGH:.3f} GFLOPs (完全独立ルーティング)")

    print("\n[Feature Importances (Gain)]")
    feature_names = [
        "1. Variance", "2. Edge Mean", "3. Edge Density", "4. Color Variance",
        "5. Extreme Pixels", "6. Total Variation", "7. Center-Surround", "8. Flatness"
    ]
    importances = clf.feature_importances_
    # パーセント表示に変換
    importances_pct = 100.0 * (importances / importances.sum())
    sorted_idx = np.argsort(importances_pct)[::-1]
    for idx in sorted_idx:
        print(f"  - {feature_names[idx]:<20}: {importances_pct[idx]:>4.1f}%")

else:
    print("\n[Result] Target accuracy could not be achieved.")

print("="*55)


In [ ]:
# --- 【方針② 最終結論】Global + Grid + Freq 厳密交差検証版 ---

import numpy as np
import json
import os
import cv2
from tqdm import tqdm
import lightgbm as lgb
import torchvision
from sklearn.model_selection import cross_val_predict

# --- 1. 初期設定 ---

BASE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(BASE_DIR, exist_ok=True)
INPUT_JSON_PATH = os.path.join(BASE_DIR, "cifar100_difficulty_labels.json")

FLOPS_LOW  = 0.301
FLOPS_HIGH = 17.6
FLOPS_ROUTER = 0.000000

# --- 2. 全特徴量の抽出関数（Global 8 + Grid 64 = 計72個） ---
def extract_ultimate_features(img_pil, grid_size=4):
    img = np.array(img_pil)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray_f = gray.astype(np.float32)
    h, w = gray.shape

    features = []
    feature_names = []

    # 【A. 全体指標】
    features.append(np.var(gray)); feature_names.append("Global_Variance")
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_mag = np.sqrt(sobelx**2 + sobely**2)
    features.append(np.mean(edge_mag)); feature_names.append("Global_EdgeMean")
    features.append(np.sum(edge_mag > 50) / gray.size); feature_names.append("Global_EdgeDensity")
    color_var = (np.var(img[:,:,0]) + np.var(img[:,:,1]) + np.var(img[:,:,2])) / 3.0
    features.append(color_var); feature_names.append("Global_ColorVar")
    features.append(np.sum((gray < 15) | (gray > 240)) / gray.size); feature_names.append("Global_ExtremePix")
    features.append(np.mean(np.abs(gray_f[:, 1:] - gray_f[:, :-1]))); feature_names.append("Global_TV")

    cy_min, cy_max, cx_min, cx_max = h//4, 3*h//4, w//4, 3*w//4
    center_area = gray[cy_min:cy_max, cx_min:cx_max]
    center_sum, total_sum = np.sum(center_area), np.sum(gray)
    surround_mean = (total_sum - center_sum) / (gray.size - center_area.size) if (gray.size - center_area.size) > 0 else 0
    features.append(abs(np.mean(center_area) - surround_mean)); feature_names.append("Global_CenterSurround")
    features.append(np.sum(np.abs(gray_f[:, 1:] - gray_f[:, :-1]) < 3) / (gray.size - h)); feature_names.append("Global_Flatness")

    # 【B. グリッド指標】
    step_h, step_w = h // grid_size, w // grid_size
    for i in range(grid_size):
        for j in range(grid_size):
            prefix = f"Grid_{i}_{j}"
            patch = gray[i*step_h:(i+1)*step_h, j*step_w:(j+1)*step_w]
            patch_edge = edge_mag[i*step_h:(i+1)*step_h, j*step_w:(j+1)*step_w]
            patch_f = gray_f[i*step_h:(i+1)*step_h, j*step_w:(j+1)*step_w]

            features.append(np.var(patch)); feature_names.append(f"{prefix}_Var")
            features.append(np.mean(patch_edge)); feature_names.append(f"{prefix}_Edge")
            tv = np.mean(np.abs(patch_f[:, 1:] - patch_f[:, :-1])) if patch_f.shape[1] > 1 else 0
            features.append(tv); feature_names.append(f"{prefix}_TV")

            a, b = patch_f[0::2, 0::2], patch_f[0::2, 1::2]
            c, d = patch_f[1::2, 0::2], patch_f[1::2, 1::2]
            if a.shape == b.shape == c.shape == d.shape and a.size > 0:
                hl = np.abs((a+c) - (b+d))
                lh = np.abs((a+b) - (c+d))
                hh = np.abs((a+d) - (b+c))
                high_freq = np.mean(hl + lh + hh)
            else:
                high_freq = 0
            features.append(high_freq); feature_names.append(f"{prefix}_HighFreq")

    return features, feature_names

# --- 3. データセット読み込み ---
print("Loading Data and Extracting 72 Ultimate Features...")
testset_raw = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=None)
with open(INPUT_JSON_PATH, 'r') as f:
    data_list = json.load(f)

acc_high_only = sum(1 for d in data_list if d["high_correct"]) / len(data_list) * 100
TARGET_ACCURACY = acc_high_only - 1.0

features_list, labels, global_feature_names = [], [], []
for idx, item in enumerate(tqdm(data_list, desc="Extracting")):
    img_pil, _ = testset_raw[item['index']]
    feats, f_names = extract_ultimate_features(img_pil, grid_size=4)
    features_list.append(feats)
    labels.append(1 if item['low_correct'] else 0)
    if idx == 0: global_feature_names = f_names

X = np.array(features_list)
y = np.array(labels)

# --- 4. 厳密な予測（Cross Validation）と、重要度算出用の全体学習 ---
print("\nRunning Cross-Validation for Strict Evaluation...")
clf_cv = lgb.LGBMClassifier(
    n_estimators=200, max_depth=6, num_leaves=63,
    learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1
)
# 未知データとしての厳密な予測確率を取得
probs = cross_val_predict(clf_cv, X, y, cv=5, method='predict_proba', n_jobs=-1)

for i, item in enumerate(data_list):
    item['rf_conf'] = probs[i][1]

print("Training Full Model for Feature Importances...")
clf_full = lgb.LGBMClassifier(
    n_estimators=200, max_depth=6, num_leaves=63,
    learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1, importance_type='gain'
)
clf_full.fit(X, y)

# --- 5. シミュレーション ---
data_list.sort(key=lambda x: x["rf_conf"], reverse=True)
total_n = len(data_list)
thresholds = np.linspace(0, 1.0, 201)

best_cost = float('inf')
best_th = None
best_acc = None

for th in thresholds:
    to_low = [d for d in data_list if d["rf_conf"] >= th]
    to_high = [d for d in data_list if d["rf_conf"] < th]

    n_low, n_high = len(to_low), len(to_high)
    avg_cost = ((total_n * FLOPS_ROUTER) + (n_low * FLOPS_LOW) + (n_high * FLOPS_HIGH)) / total_n
    accuracy = 100 * (sum(1 for d in to_low if d["low_correct"]) + sum(1 for d in to_high if d["high_correct"])) / total_n

    if accuracy >= TARGET_ACCURACY and avg_cost < best_cost:
        best_cost = avg_cost
        best_th = th
        best_acc = accuracy

# --- 6. 詳細結果の出力 ---
print("\n" + "="*60)
print("  PROPOSED ROUTER (Ultimate / Strict Cross-Validation)  ")
print("="*60)
if best_th is not None:
    n_easy_total = sum(1 for d in data_list if d["low_correct"])
    n_easy_to_low = sum(1 for d in data_list if d["low_correct"] and d["rf_conf"] >= best_th)

    print(f"[System Average Performance]")
    print(f"Router Execution Cost : ~ 0.000 GFLOPs")
    print(f"System Average Cost   : {best_cost:.3f} GFLOPs (Threshold: {best_th:.2f})")
    print(f"System Accuracy       : {best_acc:.2f}%")
    print(f"Easy Saved Ratio      : {n_easy_to_low/n_easy_total*100:.1f}%")

    print("\n[Latency Analysis (Worst-case)]")
    print(f"  Hard画像推論遅延: 17.600 GFLOPs (二重推論ゼロ)")

    print("\n[Aggregated Feature Importances (Global + Center + Edge)]")
    importances = clf_full.feature_importances_
    importances_pct = 100.0 * (importances / importances.sum())

    aggregated_imp = {}
    center_coords = [(1,1), (1,2), (2,1), (2,2)]

    for name, imp in zip(global_feature_names, importances_pct):
        if name.startswith("Global_"):
            aggregated_imp[name] = imp
        elif name.startswith("Grid_"):
            parts = name.split('_')
            r, c = int(parts[1]), int(parts[2])
            metric = parts[3]
            group_name = f"Center_Grid_{metric}" if (r, c) in center_coords else f"Edge_Grid_{metric}"
            aggregated_imp[group_name] = aggregated_imp.get(group_name, 0.0) + imp

    sorted_agg = sorted(aggregated_imp.items(), key=lambda x: x[1], reverse=True)
    for i, (name, imp) in enumerate(sorted_agg):
        print(f"  {i+1:2d}. {name:<25}: {imp:>5.2f}%")

    center_total = sum(imp for name, imp in aggregated_imp.items() if name.startswith("Center_"))
    edge_total = sum(imp for name, imp in aggregated_imp.items() if name.startswith("Edge_"))
    global_total = sum(imp for name, imp in aggregated_imp.items() if name.startswith("Global_"))

    print("\n[Area Total Contribution]")
    print(f"  - Center Grid (中央部・被写体) : {center_total:>5.2f}%")
    print(f"  - Edge Grid   (周辺部・背景)   : {edge_total:>5.2f}%")
    print(f"  - Global      (画像全体)       : {global_total:>5.2f}%")
else:
    print("\n[Result] Target accuracy could not be achieved.")
print("="*60)


In [ ]:
# --- 【背水の陣】Grid 2x2 + 強力正則化 厳密交差検証版 ---

import numpy as np
import json
import os
import cv2
from tqdm import tqdm
import lightgbm as lgb
import torchvision
from sklearn.model_selection import cross_val_predict

# --- 1. 初期設定 ---

BASE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(BASE_DIR, exist_ok=True)
INPUT_JSON_PATH = os.path.join(BASE_DIR, "cifar100_difficulty_labels.json")

FLOPS_LOW  = 0.301
FLOPS_HIGH = 17.6
FLOPS_ROUTER = 0.000000

# --- 2. 特徴量抽出関数（Grid 2x2 = 計24個に絞り込み） ---
def extract_robust_features(img_pil, grid_size=2):
    img = np.array(img_pil)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray_f = gray.astype(np.float32)
    h, w = gray.shape

    features = []

    # 【A. 全体指標 (8個)】
    features.append(np.var(gray))
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_mag = np.sqrt(sobelx**2 + sobely**2)
    features.append(np.mean(edge_mag))
    features.append(np.sum(edge_mag > 50) / gray.size)
    color_var = (np.var(img[:,:,0]) + np.var(img[:,:,1]) + np.var(img[:,:,2])) / 3.0
    features.append(color_var)
    features.append(np.sum((gray < 15) | (gray > 240)) / gray.size)
    features.append(np.mean(np.abs(gray_f[:, 1:] - gray_f[:, :-1])))

    cy_min, cy_max, cx_min, cx_max = h//4, 3*h//4, w//4, 3*w//4
    center_area = gray[cy_min:cy_max, cx_min:cx_max]
    center_sum, total_sum = np.sum(center_area), np.sum(gray)
    surround_mean = (total_sum - center_sum) / (gray.size - center_area.size) if (gray.size - center_area.size) > 0 else 0
    features.append(abs(np.mean(center_area) - surround_mean))
    features.append(np.sum(np.abs(gray_f[:, 1:] - gray_f[:, :-1]) < 3) / (gray.size - h))

    # 【B. グリッド指標 (2x2 = 16個)】
    step_h, step_w = h // grid_size, w // grid_size
    for i in range(grid_size):
        for j in range(grid_size):
            patch = gray[i*step_h:(i+1)*step_h, j*step_w:(j+1)*step_w]
            patch_edge = edge_mag[i*step_h:(i+1)*step_h, j*step_w:(j+1)*step_w]
            patch_f = gray_f[i*step_h:(i+1)*step_h, j*step_w:(j+1)*step_w]

            features.append(np.var(patch))
            features.append(np.mean(patch_edge))
            tv = np.mean(np.abs(patch_f[:, 1:] - patch_f[:, :-1])) if patch_f.shape[1] > 1 else 0
            features.append(tv)

            a, b = patch_f[0::2, 0::2], patch_f[0::2, 1::2]
            c, d = patch_f[1::2, 0::2], patch_f[1::2, 1::2]
            if a.shape == b.shape == c.shape == d.shape and a.size > 0:
                high_freq = np.mean(np.abs((a+c)-(b+d)) + np.abs((a+b)-(c+d)) + np.abs((a+d)-(b+c)))
            else:
                high_freq = 0
            features.append(high_freq)

    return features

# --- 3. データセット読み込み ---
print("Loading Data and Extracting 24 Robust Features...")
testset_raw = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=None)
with open(INPUT_JSON_PATH, 'r') as f:
    data_list = json.load(f)

acc_high_only = sum(1 for d in data_list if d["high_correct"]) / len(data_list) * 100
TARGET_ACCURACY = acc_high_only - 1.0

features_list, labels = [], []
for item in tqdm(data_list, desc="Extracting"):
    img_pil, _ = testset_raw[item['index']]
    features_list.append(extract_robust_features(img_pil, grid_size=2))
    labels.append(1 if item['low_correct'] else 0)

X = np.array(features_list)
y = np.array(labels)

# --- 4. 厳密な予測（Cross Validation + 強力正則化） ---
print("\nRunning Strict Cross-Validation with Strong Regularization...")
clf_cv = lgb.LGBMClassifier(
    n_estimators=300,        # 木の数を増やしてじっくり学習
    max_depth=6,             # FPGA制約
    num_leaves=31,           # 葉の数を減らしてシンプル化（過学習防止）
    min_child_samples=50,    # 少数のデータでの分岐を禁止（過学習防止）
    colsample_bytree=0.8,    # 特徴量のランダム間引き（汎化性能向上）
    subsample=0.8,           # データのランダム間引き（汎化性能向上）
    learning_rate=0.03,      # 学習率を下げて慎重に
    reg_alpha=0.1,           # L1正則化
    reg_lambda=0.1,          # L2正則化
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

probs = cross_val_predict(clf_cv, X, y, cv=5, method='predict_proba', n_jobs=-1)

for i, item in enumerate(data_list):
    item['rf_conf'] = probs[i][1]

# --- 5. シミュレーション ---
data_list.sort(key=lambda x: x["rf_conf"], reverse=True)
total_n = len(data_list)
thresholds = np.linspace(0, 1.0, 201)

best_cost = float('inf')
best_th = None
best_acc = None

for th in thresholds:
    to_low = [d for d in data_list if d["rf_conf"] >= th]
    to_high = [d for d in data_list if d["rf_conf"] < th]

    n_low, n_high = len(to_low), len(to_high)
    avg_cost = ((total_n * FLOPS_ROUTER) + (n_low * FLOPS_LOW) + (n_high * FLOPS_HIGH)) / total_n
    accuracy = 100 * (sum(1 for d in to_low if d["low_correct"]) + sum(1 for d in to_high if d["high_correct"])) / total_n

    if accuracy >= TARGET_ACCURACY and avg_cost < best_cost:
        best_cost = avg_cost
        best_th = th
        best_acc = accuracy

# --- 6. 詳細結果の出力 ---
print("\n" + "="*60)
print("  PROPOSED ROUTER (Robust 2x2 Grid / Strict CV)  ")
print("="*60)
if best_th is not None:
    n_easy_total = sum(1 for d in data_list if d["low_correct"])
    n_easy_to_low = sum(1 for d in data_list if d["low_correct"] and d["rf_conf"] >= best_th)

    print(f"[System Average Performance]")
    print(f"Router Execution Cost : ~ 0.000 GFLOPs")
    print(f"System Average Cost   : {best_cost:.3f} GFLOPs (Threshold: {best_th:.2f})")
    print(f"System Accuracy       : {best_acc:.2f}%")
    print(f"Easy Saved Ratio      : {n_easy_to_low/n_easy_total*100:.1f}%")
else:
    print("\n[Result] Target accuracy could not be achieved.")
print("="*60)


In [ ]:
# --- 【最終決戦】 8x8 Raw Pixel Semantic Router (厳密CV版) ---

import numpy as np
import json
import os
import cv2
from tqdm import tqdm
import lightgbm as lgb
import torchvision
from sklearn.model_selection import cross_val_predict

# --- 1. 初期設定 ---

BASE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(BASE_DIR, exist_ok=True)
INPUT_JSON_PATH = os.path.join(BASE_DIR, "cifar100_difficulty_labels.json")

FLOPS_LOW  = 0.301
FLOPS_HIGH = 17.6
FLOPS_ROUTER = 0.000000 # FPGA上のピクセル間引きとIf文のみ（DSPゼロ）

# --- 2. 究極のDSPフリー特徴量: 8x8 Raw Pixels (192次元) ---
def extract_raw_pixel_features(img_pil):
    # PIL画像をNumPy配列に変換
    img = np.array(img_pil)

    # 8x8の極小サイズにリサイズ（FPGAにおけるピクセル間引きのシミュレーション）
    # ※cv2.INTER_NEAREST は補間計算を行わない（最もFPGAの挙動に近い）
    img_tiny = cv2.resize(img, (8, 8), interpolation=cv2.INTER_NEAREST)

    # 8x8x3 = 192次元の1次元配列に平坦化
    features = img_tiny.flatten().tolist()

    return features

# --- 3. データセット読み込み ---
print("Loading Data and Extracting 192 Raw Pixel Features...")
testset_raw = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=None)
with open(INPUT_JSON_PATH, 'r') as f:
    data_list = json.load(f)

acc_high_only = sum(1 for d in data_list if d["high_correct"]) / len(data_list) * 100
TARGET_ACCURACY = acc_high_only - 1.0

features_list, labels = [], []
for item in tqdm(data_list, desc="Extracting"):
    img_pil, _ = testset_raw[item['index']]
    features_list.append(extract_raw_pixel_features(img_pil))
    labels.append(1 if item['low_correct'] else 0)

X = np.array(features_list)
y = np.array(labels)

# --- 4. 厳密な予測（Cross Validation） ---
# Rawピクセルからセマンティクスを学習させるため、木の深さと数を最大化
print("\nRunning Strict Cross-Validation (Semantic Pattern Matching)...")
clf_cv = lgb.LGBMClassifier(
    n_estimators=500,        # 複雑なパターンを覚えるために木を増やす
    max_depth=8,             # 空間パターンを捉えるため少し深めを許容 (FPGA制約内)
    num_leaves=127,
    learning_rate=0.01,      # 慎重に学習
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# 未知のデータに対する厳密な予測確率（ズルなし）
probs = cross_val_predict(clf_cv, X, y, cv=5, method='predict_proba', n_jobs=-1)

for i, item in enumerate(data_list):
    item['rf_conf'] = probs[i][1]

# --- 5. シミュレーション ---
data_list.sort(key=lambda x: x["rf_conf"], reverse=True)
total_n = len(data_list)
thresholds = np.linspace(0, 1.0, 201)

best_cost = float('inf')
best_th = None
best_acc = None

for th in thresholds:
    to_low = [d for d in data_list if d["rf_conf"] >= th]
    to_high = [d for d in data_list if d["rf_conf"] < th]

    n_low, n_high = len(to_low), len(to_high)
    avg_cost = ((total_n * FLOPS_ROUTER) + (n_low * FLOPS_LOW) + (n_high * FLOPS_HIGH)) / total_n
    accuracy = 100 * (sum(1 for d in to_low if d["low_correct"]) + sum(1 for d in to_high if d["high_correct"])) / total_n

    if accuracy >= TARGET_ACCURACY and avg_cost < best_cost:
        best_cost = avg_cost
        best_th = th
        best_acc = accuracy

# --- 6. 詳細結果の出力 ---
print("\n" + "="*60)
print("  PROPOSED ROUTER (8x8 Raw Pixels Semantic Router)  ")
print("="*60)
if best_th is not None:
    n_easy_total = sum(1 for d in data_list if d["low_correct"])
    n_easy_to_low = sum(1 for d in data_list if d["low_correct"] and d["rf_conf"] >= best_th)

    print(f"[System Average Performance]")
    print(f"Router Execution Cost : ~ 0.000 GFLOPs (Zero DSP)")
    print(f"System Average Cost   : {best_cost:.3f} GFLOPs (Threshold: {best_th:.2f})")
    print(f"System Accuracy       : {best_acc:.2f}%")
    print(f"Easy Saved Ratio      : {n_easy_to_low/n_easy_total*100:.1f}%")
else:
    print("\n[Result] Target accuracy could not be achieved.")
print("="*60)
